In [ ]:
df = pd.read_parquet('../data/processed/model_df.parquet')

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# -----------------------------
# 1. Define feature columns
# -----------------------------
# state_cols = [col for col in df.columns if col.startswith('customer_state_')]

# # Drop one state dummy as reference category
# if len(state_cols) > 0:
#     state_cols = state_cols[:-1]

features = [
    'num_prior_orders',
    'days_since_last_order',
    'customer_tenure_days',
    #'prior_total_spend',
    'avg_prior_order_value',
    #'payment_value',
    'late_delivery',
    #'delivery_delay_days'

] #+ state_cols

target = 'churn_90d'

# -----------------------------
# 2. Build modeling dataframe
# -----------------------------
model_df = df[features + [target]].copy()

# Convert bool -> int
for col in model_df.columns:
    if model_df[col].dtype == 'bool':
        model_df[col] = model_df[col].astype(int)

# Force numeric conversion
for col in model_df.columns:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')


model_df = model_df[model_df['num_prior_orders'] >= 1].copy()


X = model_df[features]
y = model_df[target].astype(int)

# Add intercept
X = sm.add_constant(X, has_constant='add')

# -----------------------------
# 4. Fit logistic regression
# -----------------------------
logit_model = sm.Logit(y, X)
result = logit_model.fit()

print(result.summary())

In [ ]:
df.groupby('late_delivery')['churn_90d'].mean().sort_values(ascending = True)

In [ ]:
feature = [
    'avg_prior_order_value',
    'late_delivery'
]
X = model_df[feature]
y = model_df[target].astype(int)

# Add intercept
X = sm.add_constant(X, has_constant='add')
# -----------------------------
# 4. Fit logistic regression
# -----------------------------
logit_testing = sm.Logit(y, X)
results = logit_testing.fit()

print(results.summary())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# -----------------------------
# 1. Define treatment, outcome, covariates
# -----------------------------
treatment = 'late_delivery'
outcome = 'churn_90d'

covariates = [
    'num_prior_orders',
    'days_since_last_order',
    'customer_tenure_days',
    'avg_prior_order_value',
    'payment_value'
]

# optional if available and clean:
# covariates += ['payment_installments']

# -----------------------------
# 2. Build causal dataframe
# -----------------------------
causal_df = df[covariates + [treatment, outcome]].copy()

for col in causal_df.columns:
    causal_df[col] = pd.to_numeric(causal_df[col], errors='coerce')

causal_df = causal_df.dropna().copy()

# keep customers with some history
causal_df = causal_df[causal_df['num_prior_orders'] >= 1].copy()

T = causal_df[treatment].astype(int).values
Y = causal_df[outcome].astype(int).values
X = causal_df[covariates].copy()

print("Causal sample shape:", causal_df.shape)
print("Treatment rate:", T.mean())
print("Outcome rate:", Y.mean())

In [ ]:
# -----------------------------
# 3. Propensity score model: P(T=1 | X)
# -----------------------------
ps_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('logit', LogisticRegression(max_iter=2000))
])

ps_model.fit(X, T)
e_hat = ps_model.predict_proba(X)[:, 1]

# clip for stability / positivity
e_hat = np.clip(e_hat, 0.01, 0.99)

print("Propensity score summary:")
print(pd.Series(e_hat).describe())

# optional diagnostic
print("Propensity AUC:", roc_auc_score(T, e_hat))

In [ ]:
# -----------------------------
# 4. Outcome models: E[Y | T=1, X] and E[Y | T=0, X]
# -----------------------------
outcome_model_treated = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('logit', LogisticRegression(max_iter=2000))
])

outcome_model_control = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('logit', LogisticRegression(max_iter=2000))
])

X_treated = X[T == 1]
Y_treated = Y[T == 1]

X_control = X[T == 0]
Y_control = Y[T == 0]

outcome_model_treated.fit(X_treated, Y_treated)
outcome_model_control.fit(X_control, Y_control)

mu1_hat = outcome_model_treated.predict_proba(X)[:, 1]
mu0_hat = outcome_model_control.predict_proba(X)[:, 1]

In [ ]:
# -----------------------------
# 5. AIPW score and ATE
# -----------------------------
aipw_scores = (
    mu1_hat - mu0_hat
    + T * (Y - mu1_hat) / e_hat
    - (1 - T) * (Y - mu0_hat) / (1 - e_hat)
)

ate_aipw = aipw_scores.mean()

print("AIPW ATE (late_delivery on churn_90d):", ate_aipw)
print("Interpretation: late delivery changes churn probability by about "
      f"{ate_aipw:.4f} ({ate_aipw*100:.2f} percentage points) on average.")

In [ ]:
# -----------------------------
# 6. Bootstrap CI
# -----------------------------
rng = np.random.default_rng(42)
B = 300
boot_ates = []

for b in range(B):
    idx = rng.choice(len(causal_df), size=len(causal_df), replace=True)

    X_b = X.iloc[idx].copy()
    T_b = T[idx]
    Y_b = Y[idx]

    ps_b = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])
    ps_b.fit(X_b, T_b)
    e_b = ps_b.predict_proba(X_b)[:, 1]
    e_b = np.clip(e_b, 0.01, 0.99)

    om1_b = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])
    om0_b = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])

    om1_b.fit(X_b[T_b == 1], Y_b[T_b == 1])
    om0_b.fit(X_b[T_b == 0], Y_b[T_b == 0])

    mu1_b = om1_b.predict_proba(X_b)[:, 1]
    mu0_b = om0_b.predict_proba(X_b)[:, 1]

    score_b = (
        mu1_b - mu0_b
        + T_b * (Y_b - mu1_b) / e_b
        - (1 - T_b) * (Y_b - mu0_b) / (1 - e_b)
    )

    boot_ates.append(score_b.mean())

ci_low, ci_high = np.percentile(boot_ates, [2.5, 97.5])

print("Bootstrap 95% CI:", (ci_low, ci_high))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(e_hat[T == 0], bins=30, alpha=0.6, label='On-time')
plt.hist(e_hat[T == 1], bins=30, alpha=0.6, label='Late')
plt.xlabel('Estimated propensity score')
plt.ylabel('Count')
plt.title('Propensity Score Overlap')
plt.legend()
plt.show()

In [ ]:
df['delay_gt_3'] = (df['delivery_delay_days'] > 3).astype(int)
df['delay_gt_7'] = (df['delivery_delay_days'] > 7).astype(int)

In [ ]:
# -----------------------------
# Subgroup definitions
# -----------------------------
causal_df['high_loyal'] = (causal_df['num_prior_orders'] >= 3).astype(int)

causal_df['recent'] = (causal_df['days_since_last_order'] <= 30).astype(int)

causal_df['high_value'] = (
    causal_df['payment_value'] > causal_df['payment_value'].median()
).astype(int)

# stronger treatment versions
causal_df['delay_gt_3'] = (df.loc[causal_df.index, 'delivery_delay_days'] > 3).astype(int)
causal_df['delay_gt_7'] = (df.loc[causal_df.index, 'delivery_delay_days'] > 7).astype(int)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np

def run_aipw(data, covariates, treatment, outcome):
    
    T = data[treatment].astype(int).values
    Y = data[outcome].astype(int).values
    X = data[covariates].copy()

    # Propensity model
    ps_model = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])
    
    ps_model.fit(X, T)
    e_hat = ps_model.predict_proba(X)[:, 1]
    e_hat = np.clip(e_hat, 0.01, 0.99)

    # Outcome models
    model_t = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])
    
    model_c = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('logit', LogisticRegression(max_iter=2000))
    ])

    model_t.fit(X[T == 1], Y[T == 1])
    model_c.fit(X[T == 0], Y[T == 0])

    mu1 = model_t.predict_proba(X)[:, 1]
    mu0 = model_c.predict_proba(X)[:, 1]

    aipw_scores = (
        mu1 - mu0
        + T * (Y - mu1) / e_hat
        - (1 - T) * (Y - mu0) / (1 - e_hat)
    )

    return aipw_scores.mean()

In [ ]:
covariates = [
    'num_prior_orders',
    'days_since_last_order',
    'customer_tenure_days',
    'avg_prior_order_value',
    'payment_value',
]

In [ ]:
print("=== Loyalty Split ===")

for group_value in [0, 1]:
    subset = causal_df[causal_df['high_loyal'] == group_value]
    
    if len(subset) < 10:
        continue
    
    loyal_ate = run_aipw(subset, covariates, 'late_delivery', 'churn_90d')
    
    print(f"high_loyal = {group_value} | n={len(subset)} | ATE={loyal_ate:.4f}")

In [ ]:
print("\n=== Recency Split ===")

for group_value in [0, 1]:
    subset = causal_df[causal_df['recent'] == group_value]
    
    ate = run_aipw(subset, covariates, 'late_delivery', 'churn_90d')
    
    print(f"recent = {group_value} | n={len(subset)} | ATE={ate:.4f}")

In [ ]:
print("\n=== Value Split ===")

for group_value in [0, 1]:
    subset = causal_df[causal_df['high_value'] == group_value]
    
    ate = run_aipw(subset, covariates, 'late_delivery', 'churn_90d')
    
    print(f"high_value = {group_value} | n={len(subset)} | ATE={ate:.4f}")

In [ ]:
print("\n=== Strong Delay (>3 days) ===")

for group_value in [0, 1]:
    subset = causal_df.copy()
    subset['treatment'] = subset['delay_gt_3']
    
    ate = run_aipw(subset, covariates, 'treatment', 'churn_90d')
    
    print(f"ATE (delay > 3 days): {ate:.4f}")

In [ ]:
print("\nNaive comparison by group:")

print(causal_df.groupby('high_loyal')[['late_delivery', 'churn_90d']].mean())

In [ ]:
causal_df['prior_total_spend'] = df.loc[causal_df.index, 'prior_total_spend']
threshold = causal_df['prior_total_spend'].quantile(0.75)

causal_df['high_spender'] = (
    causal_df['prior_total_spend'] > threshold
).astype(int)

In [ ]:
print("\n=== High Spender Split ===")

for group_value in [0, 1]:
    subset = causal_df[causal_df['high_spender'] == group_value]
    
    if len(subset) < 200:
        continue
    
    ate = run_aipw(subset, covariates, 'late_delivery', 'churn_90d')
    
    print(f"high_spender = {group_value} | n={len(subset)} | ATE={ate:.4f}")

In [ ]:
high_loyal_df = causal_df[causal_df['high_loyal'] == 1].copy()

# -----------------------------
# Basic stats
# -----------------------------
n_customers = len(high_loyal_df)
baseline_churn = high_loyal_df['churn_90d'].mean()

# overall average (for comparison)
avg_order_value_all = high_loyal_df['payment_value'].mean()

# average among churned customers (THIS is what we want)
churned_loyal = high_loyal_df[high_loyal_df['churn_90d'] == 1]
avg_order_value_churned = churned_loyal['payment_value'].mean()

print("n:", n_customers)
print("baseline churn:", baseline_churn)
print("avg order value (all loyal):", avg_order_value_all)
print("avg order value (churned loyal):", avg_order_value_churned)


# -----------------------------
# Causal effect → business impact
# -----------------------------
ATE = loyal_ate  # your estimated effect for high_loyal

# customers saved (prevented churn)
customers_saved = n_customers * ATE

# assume conservative future purchases
future_orders = 2  

# revenue saved (using churned customer value)
revenue_saved = customers_saved * avg_order_value_churned * future_orders

print("\n--- Business Impact ---")
print(f"Customers saved: {customers_saved:.0f}")
print(f"Estimated revenue saved: ${revenue_saved:,.2f}")

In [ ]:
customers_saved = n_customers * loyal_ate
print(f"{customers_saved.round()} is the total amount of customers saved")

In [ ]:
reduction_rate = 0.25 # assume we reduce delays by 25%

customers_saved_partial = customers_saved * reduction_rate
revenue_saved_partial = customers_saved_partial * avg_order_value_churned * future_orders

print("\n--- Scenario: 25% Delay Reduction ---")
print(f"Customers saved: {customers_saved_partial:.0f}")
print(f"Revenue saved: ${revenue_saved_partial:,.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# reduction rate from 0% to 100%
reduction_rates = np.linspace(0, 1, 101)

# revenue saved at each reduction rate
revenue_saved = (
    n_customers
    * ATE
    * reduction_rates
    * avg_order_value_churned
    * future_orders
)

plt.figure(figsize=(8, 5))
plt.plot(reduction_rates * 100, revenue_saved)
plt.xlabel("Reduction in Delays (%)")
plt.ylabel("Estimated Revenue Saved ($)")
plt.title("Estimated Revenue Saved vs Delay Reduction Rate")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

reduction_rates = np.linspace(0, 1, 100)

# diminishing returns function
effective_reduction = 1 - np.exp(-3 * reduction_rates)

revenue_saved = (
    n_customers
    * ATE
    * effective_reduction
    * avg_order_value_churned
    * future_orders
)

plt.figure(figsize=(8,5))
plt.plot(reduction_rates, revenue_saved)
plt.xlabel("Reduction Rate")
plt.ylabel("Estimated Revenue Saved ($)")
plt.title("Revenue Impact with Diminishing Returns")
plt.show()